## 1. Imports and configuration

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time, warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                     cross_val_score, GridSearchCV, learning_curve)
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.feature_selection import f_classif, mutual_info_classif
from sklearn.metrics import (accuracy_score, f1_score, balanced_accuracy_score,
                             precision_score, recall_score, roc_auc_score,
                             confusion_matrix, classification_report)

RANDOM_STATE = 42
sns.set_style("whitegrid")
%matplotlib inline
np.random.seed(RANDOM_STATE)

# Model factories. A lighter forest is used only inside the CV-heavy sweeps
# (component sweep, grid search, learning curve) purely for speed; the final
# reported models use the full forest.
def make_lr():        return LogisticRegression(max_iter=5000, class_weight="balanced", random_state=RANDOM_STATE)
def make_rf():        return RandomForestClassifier(n_estimators=250, min_samples_leaf=2,
                                                    class_weight="balanced_subsample", n_jobs=-1, random_state=RANDOM_STATE)
def make_rf_light():  return RandomForestClassifier(n_estimators=120, min_samples_leaf=2,
                                                    class_weight="balanced_subsample", n_jobs=-1, random_state=RANDOM_STATE)

def multiclass_scores(y_true, y_pred):
    return {"Accuracy": accuracy_score(y_true, y_pred),
            "Macro-Prec": precision_score(y_true, y_pred, average="macro"),
            "Macro-Recall": recall_score(y_true, y_pred, average="macro"),
            "Macro-F1": f1_score(y_true, y_pred, average="macro"),
            "Balanced-Acc": balanced_accuracy_score(y_true, y_pred)}

## 2. Load the dataset

The canonical source is the **UCI ML Repository, ID 270**. The official loader exposes only
the 128 features and gas labels:

```python
from ucimlrepo import fetch_ucirepo          # pip install ucimlrepo
ds = fetch_ucirepo(id=270)
X, y = ds.data.features, ds.data.targets
```

Our **sensor-drift study needs the batch index** (which acquisition month each sample came
from). We therefore read a public mirror of the same 13,910 measurements that additionally
preserves the `Batch ID` column, so the data is identical to UCI 270 but keeps the time
structure.

In [3]:
URL = ("https://raw.githubusercontent.com/HakanKARASU/"
       "Gas-Sensor-Array-Drift-Dataset/main/gas_sensor_data.csv")
df = pd.read_csv(URL)
FEATURES = [f"feature_{i}" for i in range(1, 129)]
print("Shape:", df.shape)
df[["Batch ID", "gas_label", "concentration"] + FEATURES[:3]].head()

Shape: (13910, 131)


,Batch ID,gas_label,concentration,feature_1,feature_2,feature_3
0,8,Acetaldehyde,30.0,12714.6680,2.126732,2.778869
1,8,Acetaldehyde,100.0,13124.5372,4.428956,2.751767
2,2,Acetaldehyde,50.0,4108.0312,1.333074,0.829559
3,8,Acetaldehyde,160.0,16628.4727,4.613198,3.810110
4,8,Acetaldehyde,140.0,2200.3432,1.623948,3.542865


## 3. Dataset description and audit

Recorded at the ChemoSignals Laboratory, UC San Diego, over **36 months (2008–2011)** with
**16 metal-oxide gas sensors**, each summarised by **8 features → 128 features** total.

- **Instances:** 13,910   **Features:** 128 continuous   **Target:** 6 gases
- **Batch ID:** 10 acquisition campaigns across the 36 months (used for the drift study)
- **Licence:** CC BY 4.0

*Citation: Vergara, A. et al. (2012). Gas Sensor Array Drift Dataset at Different
Concentrations. UCI ML Repository. https://doi.org/10.24432/C5MK6M*

A quick audit follows the Lab 02 pattern (types, missing values, duplicates, balance, scale).

In [4]:
audit = pd.DataFrame({
    "dtype": df[FEATURES].dtypes.astype(str).values,
}, index=FEATURES)
print("Missing values (whole table):", int(df.isna().sum().sum()))
print("Duplicate rows:", int(df.duplicated().sum()))
print("\nGas class counts:\n", df["gas_label"].value_counts())
print("\nSamples per batch:\n", df["Batch ID"].value_counts().sort_index())
print("\nFeature scale (min .. max): {:.0f} .. {:.0f}".format(
      df[FEATURES].values.min(), df[FEATURES].values.max()))
print("Most skewed features (|skew|):")
print(df[FEATURES].skew().abs().sort_values(ascending=False).head(5).round(2))

Missing values (whole table): 0
Duplicate rows: 0

Gas class counts:
 gas_label
Acetone         3009
Ethylene        2926
Ethanol         2565
Acetaldehyde    1936
Toluene         1833
Ammonia         1641
Name: count, dtype: int64

Samples per batch:
 Batch ID
1      445
2     3600
3     1244
4     1586
5      161
6      197
7     2300
8     3613
9      294
10     470
Name: count, dtype: int64

Feature scale (min .. max): -23661 .. 670687
Most skewed features (|skew|):
feature_42    117.93
feature_58    117.77
feature_50    117.28
feature_29     94.62
feature_10     88.79
dtype: float64


> **Audit read-out.** No missing values and no duplicate rows, so no imputation or
> de-duplication is required. The six classes are mildly imbalanced (so we will favour
> macro-averaged metrics), the 10 batches are very unevenly sized, and the features span
> roughly six orders of magnitude with heavy skew — which makes **standardisation essential**
> and motivates a **log transform**.